# **Preprocessing**

In [ ]:
import pandas as pd
import glob
import os

In [ ]:
files = glob.glob("/content/tweets-data/*.csv")

print("Jumlah file:", len(files))

for file in files:
    print(os.path.basename(file))

In [ ]:
dfs = []

for file in files:
    temp = pd.read_csv(file)

    temp["source_file"] = os.path.basename(file)

    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True)

In [ ]:
print("Jumlah data raw:", len(df_raw))

In [ ]:
print(df_raw.columns.tolist())

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw["full_text"].sample(20).tolist()

In [ ]:
df_raw.to_csv(
    "/content/traffic_jakarta_raw_merged.csv",
    index=False,
    encoding="utf-8-sig"
)

# **Data Cleaning**

In [ ]:
df = df_raw.copy()

In [ ]:
print("Duplicate Tweet ID:", df["id_str"].duplicated().sum())

In [ ]:
print(
    "Duplicate text:",
    df["full_text"].duplicated().sum()
)

In [ ]:
before = len(df)

df = df.drop_duplicates(
    subset=["id_str"],
    keep="first"
)

after = len(df)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

In [ ]:
before = len(df)

df = df.drop_duplicates(
    subset=["full_text"],
    keep="first"
)

after = len(df)

print("Before:", before)
print("After:", after)
print("Removed duplicate texts:", before - after)

In [ ]:
df = df.dropna(
    subset=["full_text"]
)

In [ ]:
print("Data tersisa:", len(df))

# **Basic Text Cleaning**

In [ ]:
import re

In [ ]:
import re

def basic_clean_text(text):
    text = str(text).lower()

    # Remove URL
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )

    # Remove mention
    text = re.sub(
        r'@\w+',
        ' ',
        text
    )

    # Remove hashtag symbol, keep the word
    text = re.sub(
        r'#',
        '',
        text
    )

    # Remove emoji, punctuation, and other symbols
    text = re.sub(
        r'[^\w\s]',
        ' ',
        text,
        flags=re.UNICODE
    )

    # Remove underscore
    text = re.sub(
        r'_',
        ' ',
        text
    )

    # Remove newline and extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()


df["clean_text"] = df["full_text"].apply(
    basic_clean_text
)

df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)

df = df[
    df["word_count"] >= 3
].copy()

df[
    ["full_text", "clean_text"]
].sample(20)

In [ ]:
df["clean_text"] = df["full_text"].apply(
    basic_clean_text
)

In [ ]:
df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)

In [ ]:
df["word_count"].describe()

In [ ]:
df = df[
    df["word_count"] >= 3
]

# **Filter Keywords**

In [ ]:
traffic_keywords = [
    "macet",
    "kemacetan",
    "padat",
    "tersendat",
    "lancar",
    "ramai lancar",
    "padat merayap",
    "lalin",
    "lalu lintas",
    "antrian",
    "antrean"
]

In [ ]:
def contains_traffic_keyword(text):

    return any(
        keyword in text
        for keyword in traffic_keywords
    )

In [ ]:
df["traffic_related"] = (
    df["clean_text"]
    .apply(contains_traffic_keyword)
)

In [ ]:
df["traffic_related"].value_counts()

# **Location Extraction**

In [ ]:
locations = {
    "Sudirman": ["sudirman"],
    "Kuningan": ["kuningan"],
    "Semanggi": ["semanggi"],
    "Pancoran": ["pancoran"],
    "Cawang": ["cawang"],
    "Gatot Subroto": [
        "gatot subroto",
        "gatsu"
    ],
    "Thamrin": ["thamrin"],
    "Slipi": ["slipi"],
    "Tomang": ["tomang"],
    "Kelapa Gading": [
        "kelapa gading"
    ],
    "Senayan": ["senayan"],
    "Grogol": ["grogol"],
    "Daan Mogot": ["daan mogot"],
    "Pluit": ["pluit"]
}

In [ ]:
def extract_location(text):

    for location, keywords in locations.items():

        for keyword in keywords:

            if keyword in text:
                return location

    return None

In [ ]:
df["traffic_location"] = (
    df["clean_text"]
    .apply(extract_location)
)

In [ ]:
df["traffic_location"].value_counts(
    dropna=False
)

# **Extract Traffic Condition**

In [ ]:
def extract_traffic_condition(text):

    if "padat merayap" in text:
        return "Heavy"

    elif "macet" in text or "kemacetan" in text:
        return "Congested"

    elif "tersendat" in text:
        return "Congested"

    elif "padat" in text:
        return "Dense"

    elif "ramai lancar" in text:
        return "Moderate"

    elif "lancar" in text:
        return "Smooth"

    else:
        return None

In [ ]:
df["traffic_condition"] = (
    df["clean_text"]
    .apply(extract_traffic_condition)
)

In [ ]:
df["traffic_condition"].value_counts(
    dropna=False
)

# **No Location Tweets Check**

In [ ]:
no_location = df[
    df["traffic_location"].isna()
]

no_location[
    ["full_text"]
].sample(
    min(20, len(no_location))
)

# **Suspicious Tweets Check**

In [ ]:
padat_samples = df[
    df["clean_text"].str.contains(
        "padat",
        na=False
    )
]

padat_samples[
    ["full_text"]
].sample(
    min(30, len(padat_samples))
)

In [ ]:
# Convert created_at to datetime
df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce",
    utc=True
)

# Convert UTC to Jakarta time
df["created_at_jakarta"] = (
    df["created_at"]
    .dt.tz_convert("Asia/Jakarta")
)

# Extract temporal features
df["hour"] = (
    df["created_at_jakarta"]
    .dt.hour
)

df["day_of_week"] = (
    df["created_at_jakarta"]
    .dt.day_name()
)

df["is_weekend"] = (
    df["created_at_jakarta"]
    .dt.dayofweek >= 5
)

df[
    [
        "created_at",
        "created_at_jakarta",
        "hour",
        "day_of_week",
        "is_weekend"
    ]
].head(10)

In [ ]:
print("Missing created_at:")
print(df["created_at_jakarta"].isna().sum())

print("\nHour distribution:")
print(
    df["hour"]
    .value_counts()
    .sort_index()
)

print("\nDay distribution:")
print(
    df["day_of_week"]
    .value_counts()
)

print("\nWeekend distribution:")
print(
    df["is_weekend"]
    .value_counts()
)

print("\nTraffic condition:")
print(
    df["traffic_condition"]
    .value_counts(dropna=False)
)

# **Dataset Summary**

In [ ]:
print("=== DATASET SUMMARY ===")

print(
    "Raw:",
    len(df_raw)
)

print(
    "After basic cleaning:",
    len(df)
)

print(
    "Has traffic keyword:",
    df["traffic_related"].sum()
)

print(
    "Has location:",
    df["traffic_location"].notna().sum()
)

print(
    "Has traffic condition:",
    df["traffic_condition"].notna().sum()
)

In [ ]:
df[df["traffic_related"] == False][["full_text", "clean_text"]]

In [ ]:
df[df["traffic_location"].isna()][["full_text", "clean_text"]]

In [ ]:
df[df["traffic_condition"].isna()][["full_text", "clean_text"]]

In [ ]:
df[[
    "full_text",
    "traffic_location",
    "traffic_condition"
]].sample(50)

# **Save Dataset**

In [ ]:
df.to_csv(
    "/content/traffic_jakarta_preprocessed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset saved!")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))